# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the mass spectrometry protein dataset using the `mlcroissant` library, referencing all record sets and fields by their `@id` as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll enumerate all record sets, and for each, print the available fields and columns by their `@id`. This is essential for subsequent data extraction and ensures correct referencing as per the Croissant schema.

In [ ]:
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    # List fields and their IDs if present
    if 'field' in rs:
        fields = rs['field']
        print("  Fields:")
        for f in fields:
            field_id = f if isinstance(f, str) else f.get('@id', str(f))
            print(f"    - {field_id}")
    # List columns if present
    if 'column' in rs:
        cols = rs['column']
        print("  Columns:")
        for c in cols:
            col_id = c if isinstance(c, str) else c.get('@id', str(c))
            print(f"    - {col_id}")
    print()

As an example, let’s inspect the records from the principal record set (by its `@id`). Below, we print a few records to review their basic structure. You can replace the `record_set_id` variable with any valid record set `@id` shown above to explore others.

In [ ]:
# Select the main record set @id from the overview, e.g., for a table of protein measurements
main_record_set_id = None
for rs in dataset.record_sets:
    if 'protein' in rs.get('name','').lower() or 'mass' in rs.get('name','').lower() or True:
        # Pick the first record set as example (adjust this if more detail is known)
        main_record_set_id = rs['@id']
        break
if main_record_set_id is None:
    raise Exception("No record set found")

print(f"Printing 3 records from RecordSet @id: {main_record_set_id}\n")
for i, row in enumerate(dataset.records(record_set=main_record_set_id)):
    print(row)
    if i >= 2:
        break

## 3. Data Extraction
Load all data from each record set into a Pandas DataFrame using their `@id`s for flexible programmatic access.

Data is organized by record set. All DataFrames are stored in a dictionary keyed by the record set `@id`.

In [ ]:
# Extract all record set @id values for full programmatic loading
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Load each record set into a DataFrame by @id
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id} (Rows: {len(df)})")
    except Exception as e:
        print(f"Could not load RecordSet @id: {rs_id}\n  Error: {e}")

# Example: Show columns in main_record_set_id DataFrame
if main_record_set_id in dataframes:
    print("\nColumns for main RecordSet:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group numeric protein data. All field/column references use their `@id`, as required.

Pick a numeric field and optionally a group/categorical field by inspecting columns above (adjust below as needed).

In [ ]:
# Choose numeric and group field column names by @id (from printout above). Adjust these to match your schema.
# For example, suppose main_record_set_id DataFrame columns: ['cr:protein_abundance', 'cr:accession', ...]
numeric_field_id = None
group_field_id = None

for col in dataframes[main_record_set_id].columns:
    # Try to heuristically pick a likely numeric column
    if any(k in col.lower() for k in ['abundance', 'intensity', 'mw', 'molecular_weight', 'coverage', 'count']):
        numeric_field_id = col
        break

for col in dataframes[main_record_set_id].columns:
    if any(k in col.lower() for k in ['sample', 'group', 'condition', 'type']):
        group_field_id = col
        break

if numeric_field_id is None:
    # fallback: just choose the first numeric column (try convert type)
    for col in dataframes[main_record_set_id].columns:
        try:
            # If >75% convert to float succeeds, treat as numeric
            vals = pd.to_numeric(dataframes[main_record_set_id][col], errors='coerce')
            if (vals.notna().sum() / len(vals)) > 0.75:
                numeric_field_id = col
                break
        except Exception:
            pass
if numeric_field_id is None:
    raise ValueError("Cannot find a numeric field in main record set.")

# Ensure the column is numeric type
df = dataframes[main_record_set_id].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove obvious outliers: keep only values greater than threshold
threshold = df[numeric_field_id].quantile(0.05)  # 5th percentile cutoff
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)})\n")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} (z-score):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally group by group_field_id if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
We can visualize the distribution of the selected numeric field (per sample/group if available) using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True, color='#4737a6')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field is available, plot grouped boxplot
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(
        data=filtered_df, x=group_field_id, y=numeric_field_id, palette="Set2"
    )
    plt.title(f"{numeric_field_id} distribution by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- Using the Croissant schema, we loaded all data entities by their `@id` and explored the core protein dataset.
- We examined protein abundance (or similar) distributions and normalized the data for further statistical analysis.
- The dataset is ready for downstream machine learning tasks such as classification, clustering, or biomarker discovery.

**Next steps:** You may perform differential analysis, feature engineering, or advanced visualization using the same `@id`-based references. See `mlcroissant` documentation for more advanced operations.